# 1. Clone the repo

In [ ]:
!git clone https://github.com/Asaf21S/flow-matching-as-a-layer.git
%cd flow-matching-as-a-layer
!git checkout asaf/stage3_continuation

In [ ]:
%cd /content/flow-matching-as-a-layer
!git pull --ff-only

# 2. Install dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

# 3. Session paths and imports

In [ ]:
import os
import sys
import logging
import warnings
from pathlib import Path

# Set to True to keep features and results on Drive and reuse them across sessions.
USE_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive/fmlayer")

REPO_ROOT = Path("/content/flow-matching-as-a-layer")
DATA_ROOT = Path("/content/data")   # always session-local: too big to sync

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    FEATURE_ROOT = DRIVE_ROOT / "features"
    RESULTS_ROOT = DRIVE_ROOT / "results"
else:
    FEATURE_ROOT = Path("/content/features")
    RESULTS_ROOT = Path("/content/results")

for path in (DATA_ROOT, FEATURE_ROOT, RESULTS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

os.environ["FMLAYER_DATA_ROOT"] = str(DATA_ROOT)
os.environ["FMLAYER_FEATURE_ROOT"] = str(FEATURE_ROOT)
os.environ["FMLAYER_RESULTS_ROOT"] = str(RESULTS_ROOT)
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("matplotlib").setLevel(logging.ERROR)

import torch
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("features ->", FEATURE_ROOT)
print("results  ->", RESULTS_ROOT)

Check what is already cached
Stage 3 reads the Stage 1 feature caches and reuses the Stage 1 probes. If the features are already on Drive you can skip cell 4 entirely — no dataset download is needed to train the flow.

In [ ]:
for path in sorted(FEATURE_ROOT.rglob("*.np*")):
    print(f"{path.relative_to(FEATURE_ROOT)}  {path.stat().st_size / 1e6:.1f} MB")

print("\nresults:")
for path in sorted(RESULTS_ROOT.glob("*")):
    count = len(list(path.glob("*"))) if path.is_dir() else ""
    print(f"  {path.name:<22} {count}")

# 4. Datasets, features, subsets

Only needed when the Drive cache is empty. DTD is ~625 MB, FGVC-Aircraft ~2.75 GB.

In [ ]:
from src.fmlayer.data.prepare import prepare_datasets
from src.fmlayer.features.extract import extract_all
from src.fmlayer.data.fewshot import build_all_subsets

prepare_datasets()
extract_all()
build_all_subsets()
print("Datasets, feature caches and subset indices ready.")

# 5. Component self-checks

In [ ]:
from src.fmlayer.train.checks import run_all_checks

run_all_checks()

# 6. Screen the configurations on one cell
Trains every configuration once on DTD / DINOv2 / K=full / seed 0 — the cell where the flow has the best chance (strongest encoder, most data). Use it to decide what earns a place in the full grid before spending hours on it.

In [ ]:
from src.fmlayer.train.train_fm import screen_configs

screening = screen_configs(max_epochs=300)

In [ ]:
screening_rn18 = screen_configs("resnet18", "dtd", k="full", max_epochs=300)
screening_air = screen_configs("resnet18", "aircraft", k="full", max_epochs=300)

In [ ]:
from src.fmlayer.train.train_fm import (
    default_configs, exploratory_configs, negative_control_configs,
)

for config in default_configs():
    print(f"  grid       {config.name}")
for config in exploratory_configs():
    print(f"  dropped    {config.name}")
for config in negative_control_configs():
    print(f"  control    {config.name}")

## Diagnose what the flow actually does
An accuracy number cannot distinguish "the flow does nothing" from "the flow does harm". This measures displacement and counts label flips in both directions, straight from the cached checkpoints — no retraining.

In [ ]:
from src.fmlayer.train.diagnostics import diagnose_all, print_diagnostics

print_diagnostics(diagnose_all(screening))

Read it like this:

* move near 0 with flip% near 0 -> the flow has collapsed to the identity, and its small negative delta is just approximation error.
* broken much larger than fixed -> the flow is moving points, and the moves are net-harmful.
* fixed larger than broken -> genuinely worth pursuing on the full grid.

# 7. Full grid
default_configs() is the screened set, swept over the three Stage 1 cells, K in {5, 10, full} and seeds {0, 1, 2}. Finished runs are loaded from Drive, so the cell is safe to re-run and safe to interrupt when the session dies.

In [ ]:
from src.fmlayer.train.train_fm import run_all_stage3

fm_results = run_all_stage3(max_epochs=500)

That is 8 configs x 3 cells x 3 K x 3 seeds = 216 runs. To fit one session, narrow any axis:

In [ ]:
from src.fmlayer.train.train_fm import run_all_stage3

fm_results = run_all_stage3(
    cells=(("dinov2_vits14", "dtd"),),   # one cell
    k_values=("full",),                  # one training-set size
    max_epochs=300,
)

In [ ]:
from src.fmlayer.train.train_fm import load_stage3_results

fm_results = load_stage3_results()

If it reports zero runs, the results root is not pointing at the drive folder the grid was written to. Check it before anything else:

In [ ]:
from src.fmlayer.train.train_fm import stage3_cache_summary

stage3_cache_summary()

# 8. Table, leaderboard and charts

In [ ]:
from src.fmlayer.stage3_report import make_stage3_report

report = make_stage3_report(fm_results, ablation_k="full")
report["leaderboard"]

Individual pieces:

In [ ]:
from src.fmlayer.stage3_report import stage3_table, print_stage3_table, leaderboard
from src.fmlayer.viz.stage3_charts import plot_config_ablation, plot_accuracy_vs_k

table = stage3_table(fm_results)
print_stage3_table(table, k="full")
leaderboard(table, k="full", top=5)

plot_config_ablation(table, "resnet18", "dtd", k="full")
plot_accuracy_vs_k(table, "dinov2_vits14", "dtd")          # error bars over 3 seeds

# 9. Learned dynamics and before/after, in one call
make_stage3_report writes the six charts. The nine per-run figures come from make_stage3_figures, which picks the best configuration per cell from the table, draws its dynamics next to the centroids control, and adds one before/after comparison per cell. Together these are exactly the fifteen figures docs/stage3.md references.

In [ ]:
from src.fmlayer.stage3_report import make_stage3_figures

figures = make_stage3_figures(fm_results, k="full", seed=0)
figures["chosen"]        # which configuration was drawn for each cell

# 10. Comparing two specific flows side by side
make_stage3_figures already emits one comparison per cell. To choose the pair yourself, note that plot_feature_comparison takes encoder so figures from different cells on the same dataset cannot overwrite each other:


In [ ]:
from src.fmlayer.viz.flow_viz import plot_feature_comparison

test_features, test_labels, _ = load_split(encoder, dataset, "test", FEATURE_ROOT)

winner = fm_results[f"standard_margin_n015/{encoder}/{dataset}/{k}/{seed}"]
control = fm_results[f"standard_centroids/{encoder}/{dataset}/{k}/{seed}"]

plot_feature_comparison(
    {"After margin FM": winner["fm_layer"], "After centroid FM": control["fm_layer"]},
    test_features, test_labels, meta["class_names"], dataset, encoder,
    targets=target_table_for_run(control),
    steps=12, save=True,
)

# 11. Single run, for debugging

In [ ]:
from src.fmlayer.train.train_fm import FlowConfig, run_stage3
from src.fmlayer.models.targets import MARGIN

config = FlowConfig(objective="hybrid", target_type=MARGIN, train_steps=12,
                    noise_std=0.15, cross_fit_folds=3)
result = run_stage3("resnet18", "dtd", "full", seed=0, config=config, max_epochs=300)
print(result["baseline_accuracy"], result["accuracy_by_steps"], result["delta_by_steps"])

## Removing old checkpoints

In [ ]:
import shutil

for name in ("curves_fm_stage3", "models_stage3"):
    shutil.rmtree(RESULTS_ROOT / name, ignore_errors=True)
print("stale Stage 3 checkpoints cleared")